# Parameters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/02_parameters.ipynb)

Every number that can vary—a mass, a controller gain, a box bound, a cost weight, a noise seed—lives in `.params` under one uniform protocol across all minilink objects.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. The `.params` protocol

Every object that holds numbers exposes them as `.params`, a `dict` (a nested tree on a composite object), and its equation methods take `params=`:

- `params=None` reads the object's own `.params`;
- a dictionary replaces them whole for that call, without mutating the object. It is never merged: a tree passed in is complete.

`params` holds numbers and numeric arrays only. Sizes, ports, and mode switches are structure, set by the constructor.

| Object | `.params` | Equation methods that take `params=` |
| --- | --- | --- |
| `System`, `DynamicSystem`, `StepSystem` | Flat dict of the model's numbers, e.g. `{"m", "l", "I", "gravity", "d"}` | `sys.f(x, u, t=0, params=None)`, `sys.h(x, u, t=0, params=None)`, `sys.tf(x, u, t=0, params=None)` |
| `DiagramSystem`, `SampledDiagramSystem` | `{sys_id: subsystem.params}`, each entry the block's own dict | `diagram.f(x, u, t=0, params=None)` with the complete tree |
| `Set`, `InputSet` | `BoxSet`: `{"lower", "upper"}`; `BallSet`: `{"center", "radius"}`; `SingletonSet`: `{"point"}`; `IntersectionSet`: its members' trees by class name, `{"BoxSet": ..., "BallSet": ...}`; `BoxInputSet`: its box's | `S.margin(z, t=0.0, params=None)`, `S.contains(z, t=0.0, params=None)`, `S.sample(key, n=None, params=None)` |
| `Field` | `QuadraticField`: `{"center", "P"}`; `GridField`: `{"levels", "values"}`; `MarginField`: its set's | `V.value(x, u=None, t=0.0, params=None)`, `V.gradient(x, u=None, t=0.0, params=None)` |
| `Distribution` | `Gaussian`: `{"mean", "std"}` or `{"mean", "cov"}`; `Uniform`: `{"lower", "upper"}`; `Particles`: `{"points"}` | `law.sample(key, n=None, params=None)` |
| `CostFunction` | `QuadraticCost`: `{"Q", "R", "S", "x_bar", "u_bar"}`; `TimeCost`: `{"x_bar", "radius"}`; `SumCost`: `{term_id: term.params}`; `ScaledCost`: `{"weight", "cost"}` | `cost.g(x, u, t=0.0, params=None)`, `cost.h(x, t=0.0, params=None)`, `cost.integrate(traj, *, discount_rate=0.0, params=None, t0=0.0)` |
| `PlanningProblem` | Its parts' trees by role, and its own numbers: `"plant"`, `"cost"`, `"X"`, `"U"`, `"X0"` (unless it is the singleton at `x0`), `"Xf"` (when set), `"x0"`, `"t0"`, `"discount_rate"` | `problem.realize(key=None, n=1)` draws one tree of the same shape |

A set or a cost built from a field carries the field's numbers under `"field"`: `V.as_cost().params` is `{"weight", "field"}`.

### Reading, overriding, and editing `sys.params`

In [ ]:
import numpy as np

from minilink import Pendulum

plant = Pendulum()
plant.params

In [ ]:
x = np.array([0.5, 0.0])
u = np.array([0.0])

# The model's own numbers (l = 1), then the complete tree at l = 2: plant.params is untouched
dx_default = plant.f(x, u)
dx_override = plant.f(x, u, 0.0, params={**plant.params, "l": 2.0})

dx_default, dx_override, plant.params["l"]

In [ ]:
# An edit in place changes the model's own numbers
plant.params["l"] = 2.0
plant.f(x, u)

### Differentiating with respect to parameters

Because `f` reads `params` as an argument, JAX differentiates $\dot x$ with respect to the whole dictionary at once, and returns a dictionary with the same keys. `plant.jacobian("f", "params")` is the same derivative as a verb, on either backend:

In [ ]:
import jax
import jax.numpy as jnp

plant = Pendulum()
x_jax = jnp.array([0.5, 0.0])
df_dparams = jax.jacobian(lambda params: plant.f(x_jax, jnp.zeros(1), 0.0, params))(plant.params)
df_dparams

In [ ]:
plant.jacobian("f", "params", x_bar=x, u_bar=u)

## 2. A diagram's tree

On a diagram, `.params` is `{sys_id: subsystem.params}`: each entry is the block's own dictionary, so an edit through the diagram is an edit of the block. Assigning `diagram.params = tree` takes the complete tree, every block named.

In [ ]:
from minilink import Impedance

loop = Impedance() @ plant
loop.params

In [ ]:
loop.params["plant"]["m"] = 1.5
plant.params["m"], loop.params["plant"] is plant.params

## 3. Sets, fields, distributions, and costs

These objects hold their numbers exactly as a plant does: an edit `S.params["upper"] = ...` moves the set, the named attributes (`box.upper`, `law.mean`, `cost.Q`) are views of that dictionary, and the equation methods take `params=`. A copy at other numbers is `copy.deepcopy`, then the same edit.

In [ ]:
import copy

from minilink import BoxSet, Gaussian, QuadraticCost, QuadraticField

box = BoxSet([-1.0, -1.0], [1.0, 1.0])
wider_box = copy.deepcopy(box)
wider_box.params["lower"] = np.array([-2.0, -2.0])
wider_box.params["upper"] = np.array([2.0, 2.0])
box.params, wider_box.upper, wider_box.margin(np.array([1.5, 0.0]))

In [ ]:
field = QuadraticField(np.zeros(2), np.eye(2))
shifted = {"center": np.array([1.0, 0.0]), "P": np.eye(2)}
field.value(np.array([1.0, 0.0])), field.value(np.array([1.0, 0.0]), params=shifted)

In [ ]:
law = Gaussian(mean=[0.0, 0.0], std=[0.1, 0.2])
moved_law = copy.deepcopy(law)
moved_law.params["mean"] = np.array([1.0, -1.0])
moved_law.params["std"] = np.array([0.5, 0.5])
law.params, moved_law.mean, moved_law.cov

In [ ]:
cost = QuadraticCost.from_system(plant)
cost.Q = np.diag([4.0, 1.0])
cost.params

In [ ]:
barrier = field.as_cost(weight=3.0)
barrier.params

## 4. Tools take their numbers at the build

A tool compiled from a system takes the numbers once, when it is built, and runs on them when a call passes `params=None`. Its `.params` returns a copy of that tree: the tree a call at other numbers edits and passes back, without a recompile.

| Tool | `.params` | The call that takes the edited tree |
| --- | --- | --- |
| `Simulator` | A copy of the system's tree at the build | `sim.solve(params=tree)` |
| Compiled evaluator (`sys.compile()`) | A copy of the system's tree at the compile | `evaluator.f(x, u, t, params)`, `evaluator.step(x, u, t, params, dt=dt)` |
| `ProblemEvaluator` | A copy: the loop's blocks, the problem's leaves, the free inputs | `pe.evaluate(params)`, `pe.trajectory(params)` |
| `Optimizer` | A copy: the program's parameters, the method's numbers under `"method"` | `opt.solve(params=tree)` |
| `Planner` | The problem's parts (live), `"method"` and `"seed"` | `planner.solve(params=tree)`, with `tree = copy.deepcopy(planner.params)` edited |
| `CompiledTrajectoryPlanner` | A copy of the planner's tree at the compile | `pv.solve(params=tree)` |

In [ ]:
evaluator = plant.compile()
heavier = evaluator.params
heavier["m"] = 3.0
evaluator.f(x, u), evaluator.f(x, u, 0.0, heavier), plant.params["m"]

## 5. Random blocks, laws over parameters, and `realize`

A random source block (such as `WhiteNoise`) has `is_random == True` and a `"seed"` in its `params`. On a `PlanningProblem`, `problem.params` holds numbers only; the laws over uncertain numbers are `problem.distributions`, a tree in the same shape, given as `PlanningProblem(..., distributions={"plant": {"m": Uniform(...)}})`.

| Call | Returns | Meaning |
| --- | --- | --- |
| `sys.is_random` | `bool` | Whether the block draws random numbers from a seed in its `params` |
| `sys.realize(key)` | `dict` | The block's `params` for one realization: a copy with a seed drawn from `key` (`key=None`: `seed=None`, the mean signal); a block with no randomness returns its `params` |
| `problem.distributions` | `dict` | The laws, at the paths of the numbers they draw |
| `problem.drawn_paths()` | `list[tuple[str, ...]]` | The paths of the leaves `realize` draws with a trial axis |
| `problem.realize(key=None, n=1)` | `dict` | One tree in the shape of `problem.params`: each drawn leaf `(n, *shape)`, `"x0"` `(n, nx)`, the other leaves shared; `key=None` gives every law's mean |
| `problem.sample_params(key)` | `dict` | One draw of each law, at its path (empty when nothing is drawn) |

In [ ]:
from minilink import PlanningProblem, Uniform, WhiteNoise

noise = WhiteNoise(1, psd=0.05, sample_period=0.01, seed=0)
noise.is_random, noise.params, noise.realize(42)

In [ ]:
problem = PlanningProblem(
    plant,
    x0=Uniform([-0.2, -0.2], [0.2, 0.2]),
    cost=cost,
    horizon=3.0,
    distributions={"plant": {"m": Uniform([0.8], [1.2])}},
)
list(problem.params), problem.distributions, problem.drawn_paths()

In [ ]:
# One realization (n=1, the default): the drawn leaves keep a trial axis of length 1
one_draw = problem.realize(0)
one_draw["x0"].shape, one_draw["plant"]["m"].shape, one_draw["plant"]["l"]

In [ ]:
# Three realizations, and the means (key=None)
batch_draw = problem.realize(0, n=3)
batch_draw["x0"].shape, batch_draw["plant"]["m"], problem.realize(None)["plant"]["m"]

Next, [`03_sets_fields_distributions`](03_sets_fields_distributions.ipynb): the full API of `Set`, `InputSet`, `Field`, and `Distribution`.